# Conversación persistente (P1-c)
El hilo del agente queda en disco: puedes reiniciar el kernel y seguir donde estabas, incluso con una aprobación pendiente. Aquí se prueba
con dos preguntas cortas y un reinicio en medio. Ver `docs/persistencia_y_reproducibilidad.md`.

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from dfir_copilot.agent.graph import build_agent
from dfir_copilot.agent.llm import make_llm
from dfir_copilot.cases import CaseWorkspace

ws = CaseWorkspace.open("IDOR-INVOICES-2020Q4-TZ-SANTIAGO")
real = ws.engine(timeout_s=120)
engine, ps = ws.pseudonymized(timeout_s=120)
ledger = ws.ledger(real)
agent = build_agent(engine, ledger, make_llm(), max_steps=14, checkpointer=ws.checkpointer())
display(pd.DataFrame(agent.threads()))
print("aprobaciones pendientes:", len(agent.pending()))

## 1. Primera pregunta (gasta API)
Corta a propósito: el objetivo es que quede conversación guardada.

In [ ]:
r = agent.ask("¿Cuántas cuentas distintas hay y cuáles son las tres con más peticiones? Responde en pocas líneas.")
print(r.status, "| pasos:", r.steps, "| tokens:", r.tokens, "| cortada por:", r.cut_by)
print(ps.reveal_any(r.answer) if r.answer else "(pausado: hay una hipótesis pendiente de tu aprobación)")

## 2. Reinicia el kernel
Menú **Kernel → Restart Kernel** (sin ejecutar nada más). Después ejecuta **solo la celda de arriba del todo (la de configuración)**:
debe mostrar el hilo `default` como `continuable: True`. Salta directamente a la sección 3.

## 3. Segunda pregunta, después del reinicio (gasta API)
Se apoya en lo que el modelo ya vio antes de reiniciar.

In [ ]:
r = agent.ask("De lo que acabas de ver, ¿cuál era la segunda cuenta con más peticiones y cuántas tenía? Responde en una línea.")
print(r.status, "| pasos:", r.steps, "| tokens:", r.tokens)
print(ps.reveal_any(r.answer) if r.answer else "(pausado)")

## 4. Qué hay guardado
Un solo punto por hilo (compactado). El archivo es JSON plano.

In [ ]:
path = ws.agent_dir / "threads.json"
print(path, "|", round(path.stat().st_size / 1024), "KB")
display(pd.DataFrame(agent.threads()))
print(agent.budget())

## 5. Reproducibilidad en el replay
`nondeterministic` separa las consultas que no dan lo mismo ni dos veces seguidas (empates en el corte de un `LIMIT`) de las alteradas.

In [ ]:
rep = ledger.replay([real, engine])
last = ledger.entries("replay")[-1]["data"]
print(len(rep), "consultas |", sum(bool(x["match"]) for x in rep), "coinciden")
print({k: last[k] for k in ("mismatches", "schema_drift", "nondeterministic", "skipped")})